# CardPerks LoRA Fine-Tuning

**Before running:** Runtime → Change runtime type → T4 GPU

Steps:
1. Install deps
2. Upload `data/transactions.csv`
3. Train (~15 min)
4. Download the adapter

In [ ]:
# Step 1: Install dependencies
!pip install -q -U torchao
!pip install -q transformers peft accelerate bitsandbytes datasets

In [ ]:
# Step 2: Upload transactions.csv (the TRAIN file, not the test file)
import os
from google.colab import files

os.makedirs('data', exist_ok=True)
uploaded = files.upload()  # select transactions.csv from cardperks-llm/data/

import shutil
for fname in uploaded:
    shutil.move(fname, f'data/{fname}')
    print(f'Saved to data/{fname}')

In [ ]:
# Verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Configuration
from pathlib import Path

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
DATA_PATH = Path('data/transactions.csv')
OUTPUT_DIR = Path('qwen-cardperks-lora')
CATEGORIES = ['dining', 'travel', 'groceries', 'gas', 'entertainment', 'other']
PROMPT_TEMPLATE = (
    'Classify this credit card transaction into exactly one category: '
    + ', '.join(CATEGORIES) + '.\n'
    'Transaction: {description}\n'
    'Category:'
)

In [ ]:
# Step 3a: Load and tokenize data
import csv
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

examples = []
with open(DATA_PATH) as f:
    for r in csv.DictReader(f):
        prompt = PROMPT_TEMPLATE.format(description=r['description'])
        examples.append({'text': f"{prompt} {r['category']}"})

dataset = Dataset.from_list(examples)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128, padding='max_length')

tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])
print(f'Dataset size: {len(tokenized)} examples')

In [ ]:
# Step 3b: Load base model + attach LoRA adapter
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map='auto',
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 3c: Train (~15 min on T4)
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    report_to='none',
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)
trainer.train()

In [ ]:
# Step 4a: Save the LoRA adapter
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f'Adapter saved to {OUTPUT_DIR}')

In [ ]:
# Step 4b: Zip and download the adapter
import shutil
from google.colab import files

shutil.make_archive('qwen-cardperks-lora', 'zip', '.', 'qwen-cardperks-lora')
files.download('qwen-cardperks-lora.zip')
print('Download started!')

In [ ]:
# Optional: Quick sanity check before downloading
from peft import PeftModel

def predict(description):
    prompt = PROMPT_TEMPLATE.format(description=description)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip().lower()
    return next((c for c in CATEGORIES if c in text), 'other')

tests = [
    ('TST* Chipotle San Jose CA', 'dining'),
    ('Delta Air Lines #4821', 'travel'),
    ('Trader Joes Los Angeles CA', 'groceries'),
    ('Shell Oil #7823 Austin TX', 'gas'),
    ('Netflix', 'entertainment'),
    ('Amazon.com', 'other'),
]

for desc, expected in tests:
    pred = predict(desc)
    status = '✓' if pred == expected else '✗'
    print(f'{status} "{desc}" → {pred} (expected {expected})')